In [2]:
import pandas as pd, numpy as np, json, re, ast
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings; warnings.filterwarnings('ignore')

CSV = 'test_results.csv'
df  = pd.read_csv(CSV)
for col in ['elapsed_seconds','input_tokens','output_tokens','attempts',
            'current_glucose','glucose_at_meal_time','insulin_units_recommended']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['is_safe'] = df['is_safe'].astype(str).str.lower().map({'true':True,'false':False})

def get_group(label):
    for p,g in [('NORM','NORMAL'),('HYPO','HYPO'),('HYPR','HYPER'),('LAI','LAI'),
                ('GLP1','GLP1'),('EDGE','EDGE'),('TRANS-NH','TRANS-NH'),
                ('TRANS-NR','TRANS-NR'),('TRANS-RN','TRANS-RN'),('TRANS-HN','TRANS-HN')]:
        if str(label).startswith(p): return g
    return 'OTHER'
df['group'] = df['label'].apply(get_group)
df['cost']         = df['input_tokens']/1e6*0.30 + df['output_tokens']/1e6*2.5
df['total_tokens'] = df['input_tokens'] + df['output_tokens']
GROUP_ORDER = ['NORMAL','HYPO','HYPER','LAI','GLP1','EDGE',
               'TRANS-NH','TRANS-NR','TRANS-RN','TRANS-HN']

def alert_fired(t):
    t=str(t).strip().lower()
    starts=['no medication is currently prescribed','no medication is currently required',
            'no medication recommendation needed','no oral medication','no medication is scheduled',
            'no immediate medication adjustments','no medication alert',
            'based on your current regimen, no medication','no active medications',
            'no medication required','no medication adjustments','no medication due']
    return not any(t.startswith(p) for p in starts) and 'no immediate dose of either is required' not in t
def alert_expected(row):
    lbl=str(row['label']).upper()
    for k in ['NO MEDICATIONS','NO MEDICATION','3HRS BEFORE','OUTSIDE WINDOW','61MIN',
              '2HRS BEFORE 9PM LAI','LATE NIGHT 11PM','RECENTLY ATE','NO MEAL OR INSULIN EXPECTED']:
        if k in lbl:
            return True if ('NOT TODAY' in lbl and 'GLP' in lbl) else False
    if 'NOT TODAY' in lbl: return False
    return not (str(row['oral_medication']).lower() in ['none','no'] and
                str(row['insulin']).lower() in ['no','none'] and
                str(row['long_acting_insulin']).lower() in ['no','none'] and
                str(row['glp1']).lower() in ['no','none'])
df['alert_fired']    = df['medication_alert'].apply(alert_fired)
df['alert_expected'] = df.apply(alert_expected, axis=1)
a_tp=(df['alert_expected'] & df['alert_fired']).sum()
a_tn=(~df['alert_expected'] &~df['alert_fired']).sum()
a_fp=(~df['alert_expected'] & df['alert_fired']).sum()
a_fn=(df['alert_expected'] &~df['alert_fired']).sum()
a_prec=a_tp/(a_tp+a_fp) if (a_tp+a_fp)>0 else 0
a_rec =a_tp/(a_tp+a_fn) if (a_tp+a_fn)>0 else 0
a_f1  =2*a_prec*a_rec/(a_prec+a_rec) if (a_prec+a_rec)>0 else 0
df['alert_fp'] = ~df['alert_expected'] & df['alert_fired']

def meal_fired(t):
    t=str(t).strip().lower()
    if t=='nan': return False
    return not any(p in t for p in ['no meal recommendation','no meal recommended',
        'no meal is recommended','no additional meal recommendation','not yet time for your'])
def meal_expected(row):
    lbl=str(row['label']).upper()
    if 'HYPO→HYPO' in lbl: return True
    for k in ['OUTSIDE WINDOW','LATE NIGHT 11PM','MIDNIGHT','2HRS BEFORE 9PM LAI',
              '3HRS AFTER','NO MEAL OR INSULIN EXPECTED','RECENTLY ATE','30MIN AGO',
              '3HRS BEFORE','3 HRS BEFORE','2HRS AFTER','BEFORE 9PM LAI','AT 9PM LAI']:
        if k in lbl: return False
    return True
df['meal_fired']    = df['meal_recommended'].apply(meal_fired)
df['meal_expected'] = df.apply(meal_expected, axis=1)
m_tp=(df['meal_expected'] & df['meal_fired']).sum()
m_tn=(~df['meal_expected'] &~df['meal_fired']).sum()
m_fp=(~df['meal_expected'] & df['meal_fired']).sum()
m_fn=(df['meal_expected'] &~df['meal_fired']).sum()
meal_acc=(m_tp+m_tn)/len(df)*100
df['meal_fp'] = ~df['meal_expected'] & df['meal_fired']
df['meal_fn'] =  df['meal_expected'] & ~df['meal_fired']

has_meal=df[df['meal_fired'] & df['carb_rule'].notna()].copy()
def carb_ok(row):
    rule=str(row['carb_rule']).upper(); meal=str(row['meal_recommended']).lower()
    carbs=["brown rice",'corn',"whole wheat chapati","whole wheat bread",'sweet potato','banana','blueberry',
           'oatmeal']
    fast=['fruit juice','banana','soda pop','apple juice','orange juice']
    if rule=='HIGH':    return not any(f in meal for f in carbs)
    elif rule=='LOW':   return any(f in meal for f in fast)
    elif rule=='NORMAL':return any(f in meal for f in carbs)
    return True
has_meal['carb_ok']=has_meal.apply(carb_ok, axis=1)
df = df.merge(has_meal[['test_id','carb_ok']], on='test_id', how='left')
df['carb_fail'] = df['carb_ok'] == False
carb_by_rule=has_meal.groupby('carb_rule')['carb_ok'].agg(['mean','count']).reset_index()
carb_by_rule.columns=['carb_rule','rate','n']; carb_by_rule['rate']*=100
overall_carb=has_meal['carb_ok'].mean()*100

def extract_intensity(s):
    m=re.search(r'intensity=([^|]+)',str(s),re.I)
    if not m: return None
    r=m.group(1).strip().lower()
    for k,v in [('avoid','Avoid'),('vigorous','Vigorous'),('moderate','Moderate'),('light','Light')]:
        if k in r: return v
    return None
def intensity_ok(g, intensity):
    if pd.isna(g) or intensity is None: return None
    if g<70:     return intensity=='Avoid'
    elif g<=89:  return intensity in ['Avoid','Light']
    elif g<=180: return True
    elif g<=270: return intensity in ['Light','Moderate']
    else:        return intensity=='Avoid'
df['ex_intensity']=df['exercise_recommended'].apply(extract_intensity)
df['ex_safe']=df['exercise_recommended'].str.lower().str.contains('status=ok',na=False)
df['ex_int_ok']=df.apply(lambda r: intensity_ok(r['current_glucose'],r['ex_intensity']),axis=1)
ex_int_acc=df['ex_int_ok'].dropna().mean()*100

def expected_ins(row):
    g=row['glucose_at_meal_time']
    if pd.isna(g): return None
    if g<=150: return 0
    if g<=200: return 2
    if g<=250: return 4
    if g<=300: return 6
    if g<=350: return 8
    if g<=400: return 10
    return None
df['exp_ins']=df.apply(expected_ins, axis=1)
df['ins_ok']=(df['exp_ins'].notna() & df['insulin_units_recommended'].notna() &
              (df['insulin_units_recommended']==df['exp_ins']))
df['ins_incorrect']=(df['exp_ins'].notna() & df['insulin_units_recommended'].notna() &
                     (df['insulin_units_recommended']!=df['exp_ins']))
ins_acc=df['ins_ok'].mean()*100
ins_eval=df[df['exp_ins'].notna()].copy()
ins_eval['match']=ins_eval['insulin_units_recommended']==ins_eval['exp_ins']
unit_acc=ins_eval.groupby('exp_ins')['match'].mean().mul(100).reset_index()
unit_acc.columns=['expected_units','accuracy']

grp_stats = df.groupby('group').agg(
    n=('test_id','count'), safe=('is_safe','sum'),
    lat_mean=('elapsed_seconds','mean'), lat_std=('elapsed_seconds','std'),
).reindex(GROUP_ORDER).reset_index()
grp_stats['pass_rate']  = grp_stats['safe']/grp_stats['n']*100
grp_stats['retry_rate'] = [df[df['group']==g]['attempts'].apply(lambda x:(x>1)).mean()*100
                           for g in GROUP_ORDER]

# ── Design (template-matching) ─────────────────────────────────────────────────
BG       = '#F0F2F5'; PANEL    = '#FFFFFF'; GRIDC    = '#E0E0E0'
TEXTC    = '#212121'; SUBC     = '#555555'
KPI_BLUE = '#3949AB'; KPI_GREEN= '#43A047'; KPI_SKY  = '#29B6F6'; KPI_ORANGE='#FB8C00'
SAFE_C   = '#66BB6A'; FAIL_C   = '#EF5350'
ATT1_C   = '#66BB6A'; ATT2_C   = '#FFA726'; ATT3_C   = '#EF5350'
LAT_C    = '#42A5F5'; PASS_C   = '#66BB6A'
WARN_C   = '#FFA726'

GROUP_DOT_C = {'NORMAL':KPI_GREEN,'HYPO':FAIL_C,'HYPER':KPI_ORANGE,'LAI':KPI_SKY,
               'GLP1':'#AB47BC','EDGE':'#EC407A','TRANS-NH':'#26A69A',
               'TRANS-NR':'#FF7043','TRANS-RN':'#5C6BC0','TRANS-HN':'#8D6E63'}

plt.rcParams.update({
    'font.family':'DejaVu Sans','axes.facecolor':PANEL,'figure.facecolor':BG,
    'axes.edgecolor':GRIDC,'axes.labelcolor':TEXTC,'xtick.color':TEXTC,
    'ytick.color':TEXTC,'text.color':TEXTC,'axes.titlesize':11,
    'axes.titleweight':'bold','axes.labelsize':9,'xtick.labelsize':8.5,
    'ytick.labelsize':8.5,'axes.grid':True,'grid.color':GRIDC,
    'grid.linewidth':0.6,'axes.spines.top':False,'axes.spines.right':False,
})

def kpi_tile(ax, title, value, subtitle='', color=KPI_BLUE):
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
    ax.set_facecolor(color); ax.patch.set_facecolor(color); ax.patch.set_visible(True)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.add_patch(plt.Rectangle((0,0),1,1, facecolor=color,
                 transform=ax.transAxes, zorder=0, clip_on=False))
    ax.text(0.5,0.76,title,   ha='center',va='center',color='white',
            fontsize=10.5,fontweight='bold',zorder=5)
    ax.text(0.5,0.44,value,   ha='center',va='center',color='white',
            fontsize=30,  fontweight='bold',zorder=5)
    ax.text(0.5,0.17,subtitle,ha='center',va='center',color='white',
            fontsize=9.5, alpha=0.92,zorder=5)

def style_ax(ax):
    ax.set_facecolor(PANEL)
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    for sp in ['bottom','left']: ax.spines[sp].set_edgecolor(GRIDC)
    ax.grid(color=GRIDC,linewidth=0.6,linestyle='-',alpha=0.8)

def make_table(ax, col_labels, col_data, col_widths=None, title='', header_color=KPI_BLUE):
    ax.set_facecolor(PANEL); ax.axis('off')
    if title:
        ax.set_title(title, fontsize=11, fontweight='bold', color=TEXTC, pad=8)
    if not col_data:
        ax.text(0.5,0.5,'No failures in this category ✓',
                ha='center',va='center',fontsize=11,color=SAFE_C,fontweight='bold')
        return
    tbl = ax.table(cellText=col_data, colLabels=col_labels,
                   loc='center', cellLoc='left',
                   colWidths=col_widths)
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1, 1.8)
    for (r,c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor(header_color)
            cell.get_text().set_color('white')
            cell.get_text().set_fontweight('bold')
        else:
            cell.set_facecolor('#F8F9FA' if r%2==1 else PANEL)
            cell.get_text().set_color(TEXTC)
        cell.set_edgecolor(GRIDC)
        cell.set_linewidth(0.5)

# ── Failure data ───────────────────────────────────────────────────────────────
df['label_short'] = df['label'].str.split('|').str[0].str.strip()
df['scenario']    = df['label'].str.split('|').str[1].str.strip().fillna('')

alert_fp_data = [[r['test_id'], r['group'], r['label_short'], r['scenario'][:45]]
                 for _,r in df[df['alert_fp']].iterrows()]
meal_fp_data  = [[r['test_id'], r['group'], r['label_short'], r['carb_rule']]
                 for _,r in df[df['meal_fp']].iterrows()]
meal_fn_data  = [[r['test_id'], r['group'], r['label_short'], r['carb_rule']]
                 for _,r in df[df['meal_fn']].iterrows()]
ins_inc_data  = [[r['test_id'], r['group'], r['label_short'],
                  f"{r['glucose_at_meal_time']:.1f}", int(r['exp_ins']), int(r['insulin_units_recommended'])]
                 for _,r in df[df['ins_incorrect']].iterrows()]
carb_fail_data= [[r['test_id'], r['group'], r['label_short'],
                  r['carb_rule'], f"{r['glucose_at_meal_time']:.1f}"]
                 for _,r in df[df['carb_fail']].iterrows()]



In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# DASHBOARD 1 — SYSTEM OVERVIEW & EFFICIENCY
# ══════════════════════════════════════════════════════════════════════════════
fig1 = plt.figure(figsize=(18, 16))
fig1.patch.set_facecolor(BG)
fig1.suptitle('OVERALL AI AGENT SYSTEM RESPONSE AND EFFICIENCY',
              fontsize=15, fontweight='bold', color=TEXTC, y=0.985)
gs1 = gridspec.GridSpec(3, 4, figure=fig1,
                         top=0.93, bottom=0.06, left=0.06, right=0.97,
                         hspace=0.45, wspace=0.35)

kpi_tile(fig1.add_subplot(gs1[0,0]),'Total Test Runs',str(len(df)),'',KPI_BLUE)
kpi_tile(fig1.add_subplot(gs1[0,1]),'Safety Pass Rate',
         f"{df['is_safe'].mean()*100:.1f}%",f"{int(df['is_safe'].sum())}/{len(df)} passed",KPI_GREEN)
kpi_tile(fig1.add_subplot(gs1[0,2]),'Avg Latency',
         f"{df['elapsed_seconds'].mean():.1f}s",f"P95 = {df['elapsed_seconds'].quantile(.95):.1f}s",KPI_SKY)
kpi_tile(fig1.add_subplot(gs1[0,3]),'Retry Rate',
         f"{(df['attempts']>1).mean()*100:.1f}%",f"{int((df['attempts']>1).sum())} retries",KPI_ORANGE)

# Chart 1: Retry Rate by Test Group (replaces latency distribution)
ax1 = fig1.add_subplot(gs1[1,0:2])
gs_r = grp_stats.dropna(subset=['retry_rate']).sort_values('retry_rate', ascending=True)
colors_r = [FAIL_C if v>=75 else (WARN_C if v>=40 else SAFE_C) for v in gs_r['retry_rate']]
bars_r = ax1.barh(gs_r['group'], gs_r['retry_rate'],
                  color=colors_r, height=0.6, edgecolor='white', linewidth=0.5)
ax1.axvline(50, color='grey', linestyle='--', linewidth=1.3, alpha=0.7, label='50% line')
for b,v in zip(bars_r, gs_r['retry_rate']):
    ax1.text(v+1, b.get_y()+b.get_height()/2, f'{v:.0f}%',
             va='center', fontsize=8.5, fontweight='bold', color=TEXTC)
ax1.set_xlim(0, 115)
ax1.set_title('Retry Rate by Test Group')
ax1.set_xlabel('% Runs Needing >1 Attempt')
legend_patches = [mpatches.Patch(color=FAIL_C,label='≥75%'),
                  mpatches.Patch(color=WARN_C,label='40–74%'),
                  mpatches.Patch(color=SAFE_C,label='<40%')]
ax1.legend(handles=legend_patches, fontsize=8, loc='lower right', framealpha=0.9)
style_ax(ax1); ax1.grid(axis='x'); ax1.grid(axis='y', alpha=0)

# Chart 2: Avg Latency by Test Group (±1 std)
ax2 = fig1.add_subplot(gs1[1,2:4])
gs_l = grp_stats.dropna(subset=['lat_mean']).sort_values('lat_mean', ascending=True)
y_pos = np.arange(len(gs_l))
ax2.barh(y_pos, gs_l['lat_mean'], color=LAT_C, height=0.6, edgecolor='white', linewidth=0.5)
ax2.errorbar(gs_l['lat_mean'], y_pos, xerr=gs_l['lat_std'].fillna(0),
             fmt='none', color='#1A237E', capsize=4, linewidth=1.4)
for i,v in enumerate(gs_l['lat_mean']):
    ax2.text(v+1.5, i, f'{v:.1f}s', va='center', fontsize=8.5, fontweight='bold', color=TEXTC)
ax2.set_yticks(y_pos); ax2.set_yticklabels(gs_l['group'])
ax2.set_title('Avg Latency by Test Group (±1 std)')
ax2.set_xlabel('Seconds')
style_ax(ax2); ax2.grid(axis='x'); ax2.grid(axis='y', alpha=0)

# Chart 3: Attempt Distribution
ax3 = fig1.add_subplot(gs1[2,0:2])
att = df['attempts'].value_counts().sort_index()
bars = ax3.bar(att.index, att.values,
               color=[ATT1_C,ATT2_C,ATT3_C][:len(att)],
               width=0.5, edgecolor='white', linewidth=0.6)
for bar,val in zip(bars, att.values):
    pct = val/len(df)*100
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{int(val)}\n({pct:.0f}%)', ha='center', va='bottom',
             fontsize=9, fontweight='bold', color=TEXTC)
ax3.set_title('Attempt Distribution (1=First Pass, 2+=Retry)')
ax3.set_xlabel('Number of Attempts'); ax3.set_ylabel('Count')
ax3.set_xticks([1,2,3])
style_ax(ax3)

# Chart 4: Safety Pass Rate by Test Group
ax4 = fig1.add_subplot(gs1[2,2:4])
gsp = grp_stats.dropna(subset=['pass_rate']).sort_values('pass_rate', ascending=True)
yp  = np.arange(len(gsp))
bar_c = [SAFE_C if v==100 else (WARN_C if v>=75 else FAIL_C) for v in gsp['pass_rate']]
ax4.barh(yp, gsp['pass_rate'], color=bar_c, height=0.6, edgecolor='white', linewidth=0.5)
ax4.axvline(80, color=SAFE_C, linestyle='--', linewidth=1.3, alpha=0.7, label='80% threshold')
for i,v in enumerate(gsp['pass_rate']):
    ax4.text(v+0.5, i, f'{v:.0f}%', va='center', fontsize=8.5, fontweight='bold', color=TEXTC)
ax4.set_yticks(yp); ax4.set_yticklabels(gsp['group'])
ax4.set_xlim(0,115)
ax4.set_title('Safety Pass Rate by Test Group')
ax4.set_xlabel('Pass Rate (%)')
ax4.legend(fontsize=8, framealpha=0.9)
style_ax(ax4); ax4.grid(axis='x'); ax4.grid(axis='y', alpha=0)

fig1.savefig('System efficiancy.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print("D1 done")


D1 done


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# DASHBOARD 2 — COST & TOKEN ANALYSIS  (costs in cents)
# ══════════════════════════════════════════════════════════════════════════════
fig2 = plt.figure(figsize=(18, 16))
fig2.patch.set_facecolor(BG)
fig2.suptitle('COST AND TOKEN ANALYSIS',
              fontsize=15, fontweight='bold', color=TEXTC, y=0.985)
gs2 = gridspec.GridSpec(3, 4, figure=fig2,
                         top=0.93, bottom=0.06, left=0.06, right=0.97,
                         hspace=0.45, wspace=0.35)

kpi_tile(fig2.add_subplot(gs2[0,0]),'Total Cost',
         f"${df['cost'].sum():.4f}",'70 test runs',KPI_BLUE)
kpi_tile(fig2.add_subplot(gs2[0,1]),'Avg Cost / Run',
         f"${df['cost'].mean():.5f}",f"≈ ${df['cost'].mean()*1000:.2f} per 1K runs",KPI_GREEN)
kpi_tile(fig2.add_subplot(gs2[0,2]),'Avg Input Tokens',
         f"{df['input_tokens'].mean():,.0f}",'95.4% of total tokens',KPI_SKY)
kpi_tile(fig2.add_subplot(gs2[0,3]),'Avg Output Tokens',
         f"{df['output_tokens'].mean():,.0f}",'4.6% of total tokens',KPI_ORANGE)

# Token by group
ax = fig2.add_subplot(gs2[1,0:2])
gt = df.groupby('group')[['input_tokens','output_tokens']].mean().reindex(GROUP_ORDER).dropna()
x  = np.arange(len(gt)); w=0.36
ax.bar(x-w/2, gt['input_tokens'],  w, label='Input tokens',  color=KPI_SKY,    edgecolor='white')
ax.bar(x+w/2, gt['output_tokens'], w, label='Output tokens', color=KPI_ORANGE, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(gt.index, rotation=35, ha='right', fontsize=8)
ax.set_title('Avg Token Usage by Test Group'); ax.set_ylabel('Tokens')
ax.legend(fontsize=8.5); style_ax(ax)

# Avg tokens by attempt
ax2b = fig2.add_subplot(gs2[1,2])
att_tok = df.groupby('attempts')['total_tokens'].mean()
labels_a = [f'{"1st" if k==1 else "2nd" if k==2 else "3rd"} pass' for k in att_tok.index]
bars_a = ax2b.bar(labels_a, att_tok.values,
                  color=[ATT1_C,ATT2_C,ATT3_C][:len(att_tok)],
                  edgecolor='white', width=0.5)
for b,v in zip(bars_a, att_tok.values):
    ax2b.text(b.get_x()+b.get_width()/2, v+500, f'{v:,.0f}',
              ha='center', va='bottom', fontsize=8.5, fontweight='bold', color=TEXTC)
ax2b.set_title('Avg Tokens by\nAttempt Count'); ax2b.set_ylabel('Avg Total Tokens')
style_ax(ax2b)

# Token composition donut
ax3b = fig2.add_subplot(gs2[1,3])
ax3b.set_facecolor(BG)
totals = [df['input_tokens'].sum(), df['output_tokens'].sum()]
pcts   = [t/sum(totals)*100 for t in totals]
wedges,texts,_ = ax3b.pie(
    totals,
    labels=[f'Input\n{totals[0]/1e6:.2f}M ({pcts[0]:.1f}%)',
            f'Output\n{totals[1]/1e3:.0f}K ({pcts[1]:.1f}%)'],
    colors=[KPI_SKY, KPI_ORANGE],
    wedgeprops=dict(edgecolor='white', linewidth=2, width=0.55),
    startangle=90, autopct='', labeldistance=1.18)
for t in texts: t.set_fontsize(8.5); t.set_color(TEXTC)
ax3b.set_title('Token Composition', color=TEXTC, fontweight='bold', fontsize=11, pad=8)

# Cost per run scatter — using CENTS (¢) instead of milli-$
ax4b = fig2.add_subplot(gs2[2,0:2])
for g in GROUP_ORDER:
    gdf = df[df['group']==g]
    if len(gdf):
        ax4b.scatter(gdf.index, gdf['cost']*100, color=GROUP_DOT_C[g],   # cents
                     s=55, label=g, zorder=3, alpha=0.9,
                     edgecolors='white', linewidths=0.5)
ax4b.axhline(df['cost'].mean()*100, color='black', linestyle='--', linewidth=1.3,
             label=f"Avg {df['cost'].mean()*100:.2f}¢", alpha=0.8)
ax4b.set_title('Cost per Run (cents)')
ax4b.set_xlabel('Test Index'); ax4b.set_ylabel('Cost (¢)')
ax4b.legend(fontsize=6.5, ncol=2, loc='upper right', framealpha=0.9)
style_ax(ax4b)

# Cost by group — cents
ax5b = fig2.add_subplot(gs2[2,2])
gc = df.groupby('group')['cost'].sum().reindex(GROUP_ORDER).dropna()
bars_c = ax5b.bar(gc.index, gc.values*100,     # cents
                  color=[GROUP_DOT_C[g] for g in gc.index],
                  edgecolor='white', width=0.65)
for b,v in zip(bars_c, gc.values*100):
    ax5b.text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.1f}¢',
              ha='center', va='bottom', fontsize=7.5, fontweight='bold', color=TEXTC)
ax5b.set_title('Total Cost by Group\n(cents)')
ax5b.set_ylabel('Cost (¢)')
ax5b.tick_params(axis='x', rotation=35, labelsize=7.5)
style_ax(ax5b)

# Cumulative cost — cents
ax6b = fig2.add_subplot(gs2[2,3])
df_s  = df.sort_values('test_id').reset_index(drop=True)
cumc  = df_s['cost'].cumsum()*100   # cents
ax6b.plot(df_s.index+1, cumc, color=KPI_BLUE, linewidth=2.2, zorder=3)
ax6b.fill_between(df_s.index+1, cumc, alpha=0.12, color=KPI_BLUE)
ax6b.text(0.97, 0.05, f"Total\n{df['cost'].sum()*100:.1f}¢",
          transform=ax6b.transAxes, ha='right', va='bottom',
          color=KPI_BLUE, fontsize=9, fontweight='bold')
ax6b.set_title('Cumulative Cost')
ax6b.set_xlabel('Test Run #'); ax6b.set_ylabel('Cumulative Cost (¢)')
style_ax(ax6b)

fig2.savefig('Cost and Token.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print("D2 done")


D2 done


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# DASHBOARD 3 — COMPONENT PERFORMANCE
# ══════════════════════════════════════════════════════════════════════════════
fig3 = plt.figure(figsize=(18, 18))
fig3.patch.set_facecolor(BG)
fig3.suptitle('AI SUB-AGENTS PERFORMANCE',
              fontsize=15, fontweight='bold', color=TEXTC, y=0.985)
gs3 = gridspec.GridSpec(3, 4, figure=fig3,
                         top=0.93, bottom=0.05, left=0.06, right=0.97,
                         hspace=0.50, wspace=0.35)

# KPI tiles
kpi_tile(fig3.add_subplot(gs3[0,0]),'🔔  Alert Agent',
         f"{a_f1*100:.1f}%",f"F1  |  Prec {a_prec*100:.0f}%  Rec {a_rec*100:.0f}%",KPI_BLUE)
kpi_tile(fig3.add_subplot(gs3[0,1]),'🍽️   Meal Agent',
         f"{meal_acc:.1f}%",f"Timing Acc  |  Carb {overall_carb:.0f}%",KPI_GREEN)
kpi_tile(fig3.add_subplot(gs3[0,2]),'🏃  Exercise Agent',
         f"{ex_int_acc:.1f}%",f"Intensity Acc  |  Safe {int(df['ex_safe'].sum())}",KPI_SKY)
kpi_tile(fig3.add_subplot(gs3[0,3]),'💉  Insulin Agent',
         f"{ins_acc:.1f}%",'Dosing Accuracy',KPI_ORANGE)

# Row 1: Alert confusion matrix | Carb rule compliance | Insulin accuracy by dose | Meal TP/FP/FN
ax_cm = fig3.add_subplot(gs3[1,0])
ax_cm.set_facecolor(PANEL); ax_cm.set_xlim(0,1); ax_cm.set_ylim(0,1); ax_cm.axis('off')
ax_cm.set_title('Alert: Confusion Matrix', pad=8)
for (x,y,v,lbl,c) in [
    (0.03,0.50,f'TP  {a_tp}','Correct fires','#43A047'),
    (0.52,0.50,f'TN  {a_tn}','Correct silences','#1E88E5'),
    (0.03,0.05,f'FP  {a_fp}','False alarms','#FB8C00'),
    (0.52,0.05,f'FN  {a_fn}','Missed alerts','#E53935'),
]:
    ax_cm.add_patch(FancyBboxPatch((x,y),0.43,0.41,boxstyle='round,pad=0.02',
                    facecolor=c, edgecolor='white', linewidth=1.5, alpha=0.2))
    ax_cm.add_patch(FancyBboxPatch((x,y),0.43,0.41,boxstyle='round,pad=0.02',
                    facecolor='none', edgecolor=c, linewidth=2))
    ax_cm.text(x+0.215,y+0.27,v, ha='center',va='center',fontsize=13,fontweight='bold',color=c)
    ax_cm.text(x+0.215,y+0.10,lbl,ha='center',va='center',fontsize=8,color=SUBC)

ax_cr = fig3.add_subplot(gs3[1,1])
cbr = carb_by_rule.dropna()
rc  = {'HIGH':KPI_ORANGE,'NORMAL':KPI_GREEN,'LOW':FAIL_C}
bars_cb = ax_cr.bar(cbr['carb_rule'], cbr['rate'],
                    color=[rc.get(r,KPI_SKY) for r in cbr['carb_rule']],
                    edgecolor='white', width=0.45)
ax_cr.axhline(80, color=SAFE_C, linestyle='--', linewidth=1.4, alpha=0.8, label='80% target')
ax_cr.set_ylim(0,115)
for b,(v,n) in zip(bars_cb, zip(cbr['rate'],cbr['n'])):
    ax_cr.text(b.get_x()+b.get_width()/2, v+1.5, f'{v:.0f}%\n(n={int(n)})',
               ha='center', va='bottom', fontsize=8.5, fontweight='bold', color=TEXTC)
ax_cr.set_title('Carb Rule Compliance by Type'); ax_cr.set_ylabel('Compliance %')
ax_cr.legend(fontsize=8.5); style_ax(ax_cr)

ax_ins = fig3.add_subplot(gs3[1,2])
ua = unit_acc.dropna()
cu = [SAFE_C if v>=80 else WARN_C if v>=60 else FAIL_C for v in ua['accuracy']]
bars_u = ax_ins.bar(ua['expected_units'].astype(int).astype(str)+'u',
                    ua['accuracy'], color=cu, edgecolor='white', width=0.5)
ax_ins.axhline(80, color=SAFE_C, linestyle='--', linewidth=1.4, alpha=0.8, label='80% target')
ax_ins.set_ylim(0,115)
for b,v in zip(bars_u, ua['accuracy']):
    ax_ins.text(b.get_x()+b.get_width()/2, v+1.5, f'{v:.0f}%',
                ha='center', va='bottom', fontsize=9.5, fontweight='bold', color=TEXTC)
ax_ins.set_title('Insulin Accuracy\nby Expected Dose'); ax_ins.set_xlabel('Expected Units')
ax_ins.set_ylabel('Accuracy %'); ax_ins.legend(fontsize=8.5); style_ax(ax_ins)

ax_mf = fig3.add_subplot(gs3[1,3])
meal_grp=[]
for g,gdf in df.groupby('group'):
    tp=(gdf['meal_expected'] & gdf['meal_fired']).sum()
    fp=(~gdf['meal_expected'] & gdf['meal_fired']).sum()
    fn=(gdf['meal_expected'] &~gdf['meal_fired']).sum()
    meal_grp.append({'group':g,'TP':tp,'FP':fp,'FN':fn})
mgdf=pd.DataFrame(meal_grp).set_index('group').reindex(GROUP_ORDER).reset_index().fillna(0)
xm=np.arange(len(mgdf)); wm=0.25
ax_mf.bar(xm-wm, mgdf['TP'], wm, label='TP', color=SAFE_C,    edgecolor='white')
ax_mf.bar(xm,    mgdf['FP'], wm, label='FP', color=WARN_C,    edgecolor='white')
ax_mf.bar(xm+wm, mgdf['FN'], wm, label='FN', color=FAIL_C,    edgecolor='white')
ax_mf.set_xticks(xm); ax_mf.set_xticklabels(mgdf['group'], rotation=35, ha='right', fontsize=7)
ax_mf.set_title('Meal Agent: TP / FP / FN\nby Group'); ax_mf.legend(fontsize=8.5)
style_ax(ax_mf)

# Row 2: Exercise intensity stacked | Insulin correct/incorrect by group
ax_int = fig3.add_subplot(gs3[2,0:2])
imap = {'Avoid':FAIL_C,'Light':WARN_C,'Moderate':SAFE_C,'Vigorous':KPI_SKY}
ig = (df.groupby('group')['ex_intensity']
      .value_counts(normalize=True).unstack(fill_value=0)*100)
for c in ['Avoid','Light','Moderate','Vigorous']:
    if c not in ig.columns: ig[c]=0
ig = ig.reindex(GROUP_ORDER).fillna(0)
bot = np.zeros(len(ig))
for cat,color in imap.items():
    vals=ig[cat].values
    ax_int.bar(ig.index, vals, bottom=bot, label=cat,
               color=color, edgecolor='white', linewidth=0.5, width=0.65)
    for i,(v,b) in enumerate(zip(vals,bot)):
        if v>9: ax_int.text(i, b+v/2, f'{v:.0f}%', ha='center', va='center',
                             fontsize=7.5, color='white', fontweight='bold')
    bot+=vals
ax_int.set_ylim(0,110); ax_int.tick_params(axis='x', rotation=35, labelsize=8)
ax_int.set_title('Exercise Intensity Distribution by Group'); ax_int.set_ylabel('% of Runs')
ax_int.legend(fontsize=8.5, loc='upper right', framealpha=0.9); style_ax(ax_int)

ax_ig = fig3.add_subplot(gs3[2,2:4])
ing = ins_eval.groupby('group').apply(
    lambda x: pd.Series({'Correct':x['match'].sum(),'Incorrect':(~x['match']).sum()}),
    include_groups=False).reindex(GROUP_ORDER).reset_index().fillna(0)
xi=np.arange(len(ing)); wi=0.36
ax_ig.bar(xi-wi/2, ing['Correct'],   wi, label='Correct',   color=SAFE_C, edgecolor='white')
ax_ig.bar(xi+wi/2, ing['Incorrect'], wi, label='Incorrect', color=FAIL_C, edgecolor='white')
ax_ig.set_xticks(xi); ax_ig.set_xticklabels(ing['group'], rotation=35, ha='right', fontsize=8)
ax_ig.set_title('Insulin Agent: Correct vs Incorrect by Group')
ax_ig.legend(fontsize=8.5); style_ax(ax_ig)

fig3.savefig('AI Sub-Agents performance.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print("D3 done")


D3 done


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# DASHBOARD 4 — FAILURE DETAIL TABLES
# ══════════════════════════════════════════════════════════════════════════════
fig4 = plt.figure(figsize=(18, 14))
fig4.patch.set_facecolor(BG)
fig4.suptitle('GLUCOSE COACHING SYSTEM — FAILURE DETAIL ANALYSIS',
              fontsize=15, fontweight='bold', color=TEXTC, y=0.985)
gs4 = gridspec.GridSpec(2, 2, figure=fig4,
                         top=0.93, bottom=0.05, left=0.04, right=0.97,
                         hspace=0.45, wspace=0.25)

# Table 1: Alert FP
ax4_1 = fig4.add_subplot(gs4[0,0])
make_table(ax4_1,
           ['Test ID','Group','Test Case','Scenario'],
           alert_fp_data,
           col_widths=[0.10,0.11,0.17,0.62],
           title=f'Alert Agent — False Positives  ({len(alert_fp_data)} cases)\n'
                 'Alert fired when no alert was expected',
           header_color=KPI_BLUE)

# Table 2: Meal FP + FN
ax4_2 = fig4.add_subplot(gs4[0,1])
fp_fn_data = [['FP  ❌']+r for r in meal_fp_data] + [['FN  ⚠️']+r for r in meal_fn_data]
make_table(ax4_2,
           ['Type','Test ID','Group','Test Case','Carb Rule'],
           fp_fn_data,
           col_widths=[0.10,0.10,0.12,0.22,0.46],
           title=f'Meal Agent — False Positives ({len(meal_fp_data)}) + False Negatives ({len(meal_fn_data)})\n'
                 'FP = meal fired when not expected  |  FN = meal missed when expected',
           header_color=KPI_GREEN)

# Table 3: Insulin incorrect
ax4_3 = fig4.add_subplot(gs4[1,0])
make_table(ax4_3,
           ['Test ID','Group','Test Case','Glucose@Meal','Expected U','Got U'],
           ins_inc_data,
           col_widths=[0.10,0.12,0.22,0.18,0.19,0.19],
           title=f'Insulin Agent — Incorrect Dosing  ({len(ins_inc_data)} cases)\n'
                 'Expected units vs recommended units',
           header_color=KPI_ORANGE)

# Table 4: Carb rule fail
ax4_4 = fig4.add_subplot(gs4[1,1])
make_table(ax4_4,
           ['Test ID','Group','Test Case','Carb Rule','Glucose@Meal'],
           carb_fail_data,
           col_widths=[0.10,0.12,0.26,0.18,0.34],
           title=f'Meal Agent — Carb Rule Failures  ({len(carb_fail_data)} cases)\n'
                 'Recommended meal did not match carb rule for glucose level',
           header_color='#7B1FA2')

fig4.savefig('Failure Analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print("D4 done")

D4 done
